## ENEM 2024 — Recortes geográficos,  variáveis derivadas (pós‐pré-processamento)

Objetivo: a partir do dataset tratado (nível candidato), criar variáveis derivadas usadas no dashboard/EDA e gerar recortes geográficos (MG), mantendo tipagem consistente e reduzindo categorias não utilizadas.

Entradas: DADOS_TRATADOS_2024 e RESULTADOS_TRATADOS_2024 (Parquet, saída do notebook 00-fbps-preprocessamento_2024).
Saídas:

DADOS_TRATADOS_2024_MEDIAS e RESULTADOS_TRATADOS_2024_MEDIAS (Parquet)  — arquivo original com as duas variáveis derivadas adicionadas 

DADOS_TRAT_MG_2024 e RESULTADOS_TRAT_MG_2024  (Parquet) — candidatos com prova em MG + variável regiao

(opcional/etapa posterior) tabelas agregadas por município/região (quando você criar/rodar df_agg)

Observação (GitHub): microdados brutos não são versionados; esta etapa opera sobre Parquet já tratado.

In [1]:
import sys
from pathlib import Path

# Permite importar o pacote `src/` a partir do diretório do projeto.
ROOT_PATH = Path().resolve().parents[1]  # notebooks/00_preprocessamento -> projeto
if str(ROOT_PATH) not in sys.path:
    sys.path.append(str(ROOT_PATH))

import re
import numpy as np
import pandas as pd


from src.config import (
    DADOS_TRATADOS_2024, 
    RESULTADOS_TRATADOS_2024,
    DADOS_TRATADOS_MEDIAS_2024,
    RESULTADOS_TRATADOS_MEDIAS_2024,
    DADOS_TRAT_MG_2024, 
    RESULTADOS_TRAT_MG_2024, 
)

from src.preprocessamento.regioes_mg import  atribuir_regiao, MAP_NOME_REGIAO, MAP_REGIOES
from src.preprocessamento.categorias import MAP_SAL_MIN_RENDA_MEDIA


pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")



### 2) Leitura do dataset tratado

In [2]:
df_d = pd.read_parquet(DADOS_TRATADOS_2024)
df_r = pd.read_parquet(RESULTADOS_TRATADOS_2024)

df_d.head(3)

,faixa_etaria,sexo,estado_civil,cor_raca,municipio,uf,escolaridade_pai,escolaridade_mae,ocup_pai,ocup_mae,n_pessoas_resd,sal_min,emp_domst,gelad,lv_rp,tv,comptdr,cel,escola,indice_consumo
0,20-25,feminino,solteiro,branca,Porto Alegre,RS,superior,superior,manual/tec,médio/tec,4,1 a 3,0,1,1,3,1,4,pública,0.66
1,26-35,feminino,solteiro,branca,São Luiz Gonzaga,RS,até fund,até fund,rural,médio/tec,2,1 a 3,0,1,0,1,0,2,pública,0.25
2,26-35,feminino,solteiro,branca,Sarandi,RS,desconhecida,superior,desconhecida,médio/tec,5,5 a 10,0,2,1,3,0,3,pública,0.70


### 3) Variáveis derivadas

Nesta etapa são criadas variáveis auxiliares que facilitam análises exploratórias, agregações e etapas posteriores do pipeline, incluindo o dashboard e a modelagem preditiva. Essas variáveis não fazem parte dos microdados originais do ENEM, mas são derivadas de campos já existentes para melhorar a interpretabilidade e a usabilidade dos dados.

#### renda_media (proxy numérica de sal_min)

A variável sal_min representa a renda familiar mensal em faixas de salários mínimos, sendo portanto uma categoria ordinal. Para permitir análises quantitativas simples — como correlações, cálculos de médias, rankings e agregações — foi criada a variável renda_media, que associa a cada faixa um valor numérico representativo, aproximando o ponto médio de cada intervalo.

Essa transformação não pretende estimar a renda real do candidato, mas apenas fornecer uma proxy numérica consistente para análises estatísticas e comparações entre grupos.

#### nota_media (indicador sintético de desempenho)

Para obter uma medida sintética do desempenho do candidato no ENEM, foi criada a variável nota_media, definida como a média aritmética das cinco notas do exame:

* Ciências da Natureza (nota_cn)
* Ciências Humanas (nota_ch)
* Linguagens (nota_lc)
* Matemática (nota_mt)
* Redação (nota_redacao)

Essa variável fornece uma visão geral do desempenho global do candidato, sendo útil para:

* comparações entre grupos socioeconômicos
* análises regionais
* visualizações no dashboard
* definição de variável alvo em experimentos de modelagem preditiva

Embora simplifique o desempenho em um único indicador, a média preserva a escala das notas individuais e facilita análises agregadas mais diretas.

##### Observação metodológica

As notas do ENEM são apresentadas em uma escala comparável entre áreas, o que permite a construção de uma medida sintética de desempenho para fins analíticos. Assim, nota_media não substitui análises específicas por área do conhecimento, mas funciona como um indicador global adequado ao objetivo deste projeto, que é investigar padrões socioeconômicos associados ao desempenho educacional.

In [3]:
#renda média

df_d["renda_media"] = (
    df_d["sal_min"].astype("string").map(MAP_SAL_MIN_RENDA_MEDIA).astype("float32")
)

In [4]:
#nota média
df_r["nota_media"] = df_r[
    ["nota_cn", "nota_ch", "nota_lc", "nota_mt", "nota_redacao"]
].mean(axis=1).astype("float32")

### 4) Recortes geográficos e limpeza de categorias

In [6]:
#MG
df_d_mg = df_d.loc[df_d['uf'] == 'MG', :].copy()
df_d_mg['uf'] = df_d_mg['uf'].cat.remove_unused_categories()

df_r_mg = df_r.loc[df_r['uf'] == 'MG', :].copy()
df_r_mg['uf'] = df_r_mg['uf'].cat.remove_unused_categories()

df_d_mg.head(3)

,faixa_etaria,sexo,estado_civil,cor_raca,municipio,uf,escolaridade_pai,escolaridade_mae,ocup_pai,ocup_mae,n_pessoas_resd,sal_min,emp_domst,gelad,lv_rp,tv,comptdr,cel,escola,indice_consumo,renda_media
17,até 19,masculino,solteiro,negra,Ribeirão das Neves,MG,médio,pós-grad,desconhecida,médio/tec,4,1 a 3,0,1,0,1,0,1,pública,0.18,2.00
34,26-35,feminino,solteiro,branca,Betim,MG,desconhecida,médio,desconhecida,básico,2,1 a 3,0,1,1,1,1,2,pública,0.39,2.00
35,46-60,feminino,casado/mora com companheiro,branca,Uberaba,MG,até fund,até fund,básico,básico,3,1 a 3,0,1,1,2,0,3,pública,0.48,2.00


### 5) Variável regiao (macro-regiões dentro de MG)


#### 5.1) Atribuição de região e mapeamento dos nomes para versões mais amigáveis


In [7]:
# Aplicar no dataframe
df_d_mg["regiao"] = df_d_mg["municipio"].astype("string").apply(atribuir_regiao)
df_d_mg["regiao"] = df_d_mg["regiao"].map(MAP_NOME_REGIAO).astype("category")


df_r_mg["regiao"] = df_r_mg["municipio"].astype("string").apply(atribuir_regiao)
df_r_mg["regiao"] = df_r_mg["regiao"].map(MAP_NOME_REGIAO).astype("category")

df_r_mg.head(3)

,uf,escola,municipio,nota_cn,nota_ch,nota_lc,nota_mt,lingua,nota_redacao,nota_media,regiao
8,MG,não informada,Sabará,NaN,NaN,NaN,NaN,espanhol,NaN,NaN,Metrop. de Belo Horizonte
67,MG,privada,São João Nepomuceno,601.50,635.60,590.30,777.30,inglês,920.00,704.94,Sudoeste de Minas
70,MG,não informada,Montes Claros,586.10,624.60,562.80,614.00,inglês,940.00,665.50,Norte de Minas


#### 5.2) Diagnóstico: municípios não classificados

In [8]:
todas = set()
for cidades in MAP_REGIOES.values():
    todas.update(cidades)

municipios_df_d = df_d_mg["municipio"].astype("string").str.strip().unique()
nao_classificados = [m for m in municipios_df_d if m not in todas]

print("Municípios únicos em MG:", len(municipios_df_d))
print("Não classificados:", len(nao_classificados))
# se quiser, mostre só os primeiros 30 para não poluir o notebook
print("Exemplos:", nao_classificados[:30])

Municípios únicos em MG: 188
Não classificados: 0
Exemplos: []


In [9]:
todas = set()
for cidades in MAP_REGIOES.values():
    todas.update(cidades)

municipios_df_r = df_r_mg["municipio"].astype("string").str.strip().unique()
nao_classificados = [m for m in municipios_df_r if m not in todas]

print("Municípios únicos em MG:", len(municipios_df_r))
print("Não classificados:", len(nao_classificados))
# se quiser, mostre só os primeiros 30 para não poluir o notebook
print("Exemplos:", nao_classificados[:30])

Municípios únicos em MG: 188
Não classificados: 0
Exemplos: []


### Exportação
Ao final desta etapa, são salvos três arquivos:

* a base completa com variáveis derivadas (DADOS_TRATADOS_MEDIAS_2024 e RESULTADOS_TRATADOS_MEDIAS_2024);
* o recorte de Minas Gerais com variável regional (DADOS_TRAT_MG_2024 e RESULTADOS_TRAT_MG_2024);

Esses arquivos servem de entrada para as etapas posteriores de consolidação longitudinal, dashboard e modelagem.

In [10]:
df_d_mg.to_parquet(DADOS_TRAT_MG_2024, index=False)  
df_d.to_parquet(DADOS_TRATADOS_MEDIAS_2024, index=False)

print("✅ Salvo base completa com médias:", DADOS_TRATADOS_MEDIAS_2024, "| shape:", df_d.shape)
print("✅ Salvo recorte MG:", DADOS_TRAT_MG_2024, "| shape:", df_d_mg.shape)

✅ Salvo base completa com médias: E:\ciencias_dados\projetos\projeto_enem_ml\dados\intermediarios\dados_tratados_2024_medias.parquet | shape: (4332944, 21)
✅ Salvo recorte MG: E:\ciencias_dados\projetos\projeto_enem_ml\dados\intermediarios\dados_tratados_MG_24.parquet | shape: (393824, 22)
✅ Salvo recorte Caxambu: E:\ciencias_dados\projetos\projeto_enem_ml\dados\intermediarios\dados_tratados_CAX_24.parquet | shape: (1082, 21)


In [11]:
df_r_mg.to_parquet(RESULTADOS_TRAT_MG_2024, index=False)  
df_r.to_parquet(RESULTADOS_TRATADOS_MEDIAS_2024, index=False)

print("✅ Salvo base completa com médias:", RESULTADOS_TRATADOS_MEDIAS_2024, "| shape:", df_r.shape)
print("✅ Salvo recorte MG:", RESULTADOS_TRAT_MG_2024, "| shape:", df_r_mg.shape)

✅ Salvo base completa com médias: E:\ciencias_dados\projetos\projeto_enem_ml\dados\intermediarios\resultados_tratados_2024_medias.parquet | shape: (4332944, 10)
✅ Salvo recorte MG: E:\ciencias_dados\projetos\projeto_enem_ml\dados\intermediarios\resultados_tratados_MG_24.parquet | shape: (393824, 11)
✅ Salvo recorte Caxambu: E:\ciencias_dados\projetos\projeto_enem_ml\dados\intermediarios\resultados_tratados_CAX_24.parquet | shape: (1082, 10)
